In [1]:
!pip install -q gradio transformers peft datasets bitsandbytes accelerate jsonlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 41.0 MB/s eta 0:00:00


In [ ]:
# For use in Colab, for local use set the path for the final model in LORA_MODEL_PATH
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from pathlib import Path

In [ ]:
# Loading model (replace model path with your local path if not using Colab)
BASE_MODEL = "mistralai/Mistral-7B-v0.1"
LORA_MODEL_PATH = "/content/drive/MyDrive/OcelotBotV2/lora-model-output/final_model"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Loading LoRA adapters...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_PATH)
model.eval()

Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading LoRA adapters...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [6]:
def generate_response(
    user_message,
    sentiment,
    temperature,
    max_tokens,
    conversation_history
):
    """
    Generate a response based on user input and sentiment.

    Args:
        user_message: User's input text
        sentiment: Selected sentiment (Positive/Neutral/Negative)
        temperature: Randomness (0.1-1.0)
        max_tokens: Maximum response length
        conversation_history: List of [user, assistant] message pairs

    Returns:
        Updated conversation history
    """
    if not user_message.strip():
        return conversation_history

    # Construct prompt with sentiment conditioning
    prompt = f"""[SENTIMENT: {sentiment.upper()}]
      [INSTRUCTION] Respond empathetically to a student, matching the tone and context of the conversation.
      [CONTEXT] {user_message}
      [RESPONSE]
      """

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    # Decode
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract response
    response = full_output.split("[RESPONSE]")[-1].strip()

    # Update conversation history
    conversation_history.append([user_message, response])

    return conversation_history

In [7]:
# Gradio setup
custom_css = """
#component-0 {
    max-width: 900px;
    margin: auto;
}
.sentiment-box {
    background-color: #f0f0f0;
    padding: 10px;
    border-radius: 5px;
    margin: 10px 0;
}
"""

with gr.Blocks(css=custom_css, title="Student Support AI") as demo:

    gr.Markdown("""
    # 🎓 Student Support AI Assistant

    This AI has been fine-tuned to provide **tone-aware, empathetic responses** to students.

    ### How to use:
    1. **Select the sentiment** of your message (the AI will match its tone accordingly)
    2. **Type your message** and press Enter or click Submit
    3. **Adjust settings** to control response style

    *The sentiment selector allows you to control how empathetic/supportive the AI's response should be.*
    """)

    with gr.Row():
        with gr.Column(scale=2):
            # Chat interface
            chatbot = gr.Chatbot(
                label="Conversation",
                height=400,
                show_label=True,
                avatar_images=(None, "🤖")
            )

            with gr.Row():
                msg = gr.Textbox(
                    label="Your message",
                    placeholder="Type your message here... (e.g., 'I'm stressed about midterms')",
                    lines=2,
                    scale=4
                )
                submit = gr.Button("Send", variant="primary", scale=1)

            with gr.Row():
                clear = gr.Button("Clear Conversation")

        with gr.Column(scale=1):
            # Controls
            gr.Markdown("### ⚙️ Settings")

            sentiment = gr.Radio(
                choices=["Negative", "Neutral", "Positive"],
                value="Neutral",
                label="Message Sentiment",
                info="How should the AI perceive your emotional state?"
            )

            gr.Markdown("---")

            temperature = gr.Slider(
                minimum=0.1,
                maximum=1.0,
                value=0.7,
                step=0.1,
                label="Creativity",
                info="Higher = more creative, Lower = more focused"
            )

            max_tokens = gr.Slider(
                minimum=50,
                maximum=300,
                value=150,
                step=10,
                label="Response Length",
                info="Maximum tokens in response"
            )

            gr.Markdown("---")

            # Example buttons
            gr.Markdown("### 💡 Try These Examples:")

            examples = gr.Examples(
                examples=[
                    ["I'm really stressed about my midterms coming up", "Negative"],
                    ["I just got an A on my research paper!", "Positive"],
                    ["What time does the library close?", "Neutral"],
                    ["I feel like I'm falling behind in all my classes", "Negative"],
                    ["My professor gave me great feedback today", "Positive"],
                ],
                inputs=[msg, sentiment],
                label=None
            )

    # Event handlers
    def user_submit(user_message, sentiment_val, temp, max_tok, history):
        """Handle user message submission"""
        return generate_response(user_message, sentiment_val, temp, max_tok, history)

    def clear_conversation():
        """Clear the conversation history"""
        return []

    # Submit on Enter or button click
    submit.click(
        user_submit,
        inputs=[msg, sentiment, temperature, max_tokens, chatbot],
        outputs=[chatbot]
    ).then(
        lambda: "",  # Clear input box after sending
        outputs=[msg]
    )

    msg.submit(
        user_submit,
        inputs=[msg, sentiment, temperature, max_tokens, chatbot],
        outputs=[chatbot]
    ).then(
        lambda: "",
        outputs=[msg]
    )

    clear.click(clear_conversation, outputs=[chatbot])

    gr.Markdown("""
    ---
    ### About This Model

    - **Base Model**: Mistral-7B-v0.1
    - **Fine-tuning**: LoRA (Low-Rank Adaptation)
    - **Training Data**: 5,000+ student conversation examples
    - **Special Feature**: Sentiment-conditioned responses (tone-aware)

    *This model was trained to understand student mental health and academic concerns,
    providing empathetic, context-appropriate responses.*
    """)

/tmp/ipython-input-3068174474.py:15: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="Student Support AI") as demo:
/tmp/ipython-input-3068174474.py:33: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipython-input-3068174474.py:33: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


In [ ]:
demo.launch(
    share=False,  # To create a public URL set to True
    debug=True,
    show_error=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>